In [144]:
import pandas as pd
import numpy as np
import datetime
import json
import os
from IPython.display import display_html
from IPython.display import HTML
import csv

In [145]:
def display_horizontal(*args):
    html_str = ''
    for df in args:
        html_str += df.to_html()
    display_html(
        html_str.replace('table','table style="display:inline;margin-right:50px"'), 
        raw=True
    )

In [146]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)
    
with open("INDEX_NAME.json", 'r', encoding = 'utf-8') as f:
    INDEX_NAME = json.load(f)

In [150]:
df = pd.read_csv("A股指数.csv")
a_share_dict = dict(zip(df["Ticker"], df["Name"]))

df = pd.read_csv("海外指数.csv")
other_market_dict = dict(zip(df["Ticker"], df["Name"]))

print(len(a_share_dict), len(other_market_dict))

333 57


In [151]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)

In [152]:
UNIVERSE_A = {}
UNIVERSE_O = {}
target_dates = {}

for ticker in INDEX_DATA:
    if ticker in a_share_dict:
        target_dates[ticker] = INDEX_DATA[ticker].index[-1]
        UNIVERSE_A[ticker] = INDEX_DATA[ticker]

    elif ticker in other_market_dict:
        target_dates[ticker] = INDEX_DATA[ticker].index[-1]
        UNIVERSE_O[ticker] = INDEX_DATA[ticker]

In [153]:
print(f"Assets under Analysis: {len(UNIVERSE_A)}  {len(UNIVERSE_O)}")
print(f"Total Assets: {len(INDEX_DATA)}")

Assets under Analysis: 73  35
Total Assets: 186


In [154]:
n = 0
for ticker in other_market_dict:
    if ticker in INDEX_DATA:
        n += 1

n

39

<br><br><br><br>

## Z Score Calculation

In [179]:
UNIVERSE = UNIVERSE_A
Z_PE_TTM = {}
Z_DIVIDENDYIELD = {}
years = 0.5
duration = int(years * 252)

for ticker in UNIVERSE:
    target_date = target_dates[ticker]
    if len(UNIVERSE[ticker]) < duration:
        continue
    
    # PE TTM 分位值
    sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
    
    Z_PE_TTM[ticker] = Z
    
    
    # 股息率 分位值
    sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
    
    Z_DIVIDENDYIELD[ticker] = Z

In [180]:
display_number = 15

df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', 'F1'])
df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', 'F2'])

# Sort the dataframes
df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='F1', ascending=True)
df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='F2', ascending=False)

# Display the top 10 sorted rows
top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

top_Z_PE_TTM.loc[:, "指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "F1"]]

top_Z_DIVIDENDYIELD.loc[:, "指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "F2"]]

In [181]:
print()
print("A股")
print(f"{INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d')}")
print(f"{years}年")
display_horizontal(top_Z_PE_TTM, top_Z_DIVIDENDYIELD)
print()


A股
2025-05-28
0.5年


,Ticker,指数名称,F1
62,930632.CSI,CS稀金属,-2.311867
35,931865.CSI,中证半导,-2.131586
46,931892.CSI,有色矿业,-2.012839
45,932066.CSI,半导体行业精选,-1.972357
32,930598.CSI,稀土产业,-1.933079
18,000811.CSI,细分有色,-1.913257
49,931461.CSI,电子50,-1.861059
5,399975.SZ,证券,-1.808051
50,931412.CSI,证券公司30,-1.730527
58,931009.CSI,建筑材料,-1.686755


In [182]:
UNIVERSE = UNIVERSE_O
Z_PE_TTM = {}
Z_DIVIDENDYIELD = {}
years = 0.5
duration = int(years * 252)

for ticker in UNIVERSE:
    target_date = target_dates[ticker]
    if len(UNIVERSE[ticker]) < duration:
        continue
    
    # PE TTM 分位值
    sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
    sample_data.dropna(inplace = True)
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
    
    Z_PE_TTM[ticker] = Z
    
    
    # 股息率 分位值
    sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
    sample_data.dropna(inplace = True)
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
    
    Z_DIVIDENDYIELD[ticker] = Z

C:\Users\Bill Yin\AppData\Local\Temp\ipykernel_35864\3217123698.py:20: RuntimeWarning: invalid value encountered in scalar divide
  Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std


In [183]:
display_number = 15

df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', 'F1'])
df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', 'F2'])

# Sort the dataframes
df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='F1', ascending=True)
df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='F2', ascending=False)

# Display the top 10 sorted rows
top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

top_Z_PE_TTM.loc[:, "指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "F1"]]

top_Z_DIVIDENDYIELD.loc[:, "指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "F2"]]

In [184]:
print()
print("海外")
print(f"{INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d')}")
print(f"{years}年")
display_horizontal(top_Z_PE_TTM, top_Z_DIVIDENDYIELD)
print()


海外
2025-05-28
0.5年


,Ticker,指数名称,F1
8,HXC.GI,纳斯达克中国金龙,-1.579310
29,930709.CSI,香港证券,-1.557096
10,931787CNY00.CSI,港股创新药,-1.115527
31,987008.CNI,港股通科技30,-1.004411
11,DJI.GI,道琼斯,-0.808026
26,930604.CSI,中概互联,-0.795803
4,HSTECH.HI,恒生科技基金,-0.757642
12,750108.MI,美国50,-0.686550
17,HSSCNE.HI,恒生新经济,-0.519620
0,NDX.GI,纳斯达克指数,-0.479953


In [114]:
ticker

'000913.SH'

In [110]:
temp = INDEX_DATA[ticker]

In [111]:
temp = temp[~temp.index.duplicated(keep = "last")]

In [113]:
temp.tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-05-09,28.068100,1.9990,7848.4305
2025-05-12,27.993900,1.8358,7818.1952
2025-05-13,28.240000,1.8197,7891.3630
2025-05-14,28.418400,1.8083,7959.5245
2025-05-15,28.409901,1.8086,7968.7475


In [143]:
INDEX_DATA['931454.CSI'].tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-05-21,21.184799,1.5253,1610.1203
2025-05-22,20.902800,1.5868,1587.0903
2025-05-23,20.940800,1.5551,1588.6814
2025-05-26,20.300900,1.5434,1545.7058
2025-05-27,20.417400,1.5453,1555.8586
